# 🏠 House Prices Prediction — chạy toàn bộ pipeline

Notebook này là điểm vào chính của dự án, tương đương `python main.py`.

**Luồng:** Clean → Feature → Preprocess → 5-fold OOF từng model → Ensemble → `submission.csv`

Trước khi chạy lần đầu:

```
pip install -r requirements.txt
pip install -e .
```

và đặt `train.csv`, `test.csv` từ Kaggle vào `data/raw/`.

In [ ]:
# ===== Thiết lập =====
import sys, warnings
from pathlib import Path

# Notebook này nằm ở gốc dự án nên import được ngay.
# Dòng dưới là phương án dự phòng nếu bạn chưa chạy `pip install -e .`
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Tự nạp lại code khi file trong src/ thay đổi.
# Thiếu hai dòng này thì bạn sửa src/ mà notebook vẫn chạy bản cũ.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100)
warnings.filterwarnings('ignore', category=FutureWarning)

print('Thư mục dự án:', ROOT)

## Tham số chạy

Sửa ba biến dưới đây thay cho các cờ dòng lệnh của `main.py`.

In [ ]:
MEMBER = 'team'          # tên bạn, để log ghi riêng cho từng người
CONFIG = None            # None = config/global_config.yaml
                         # hoặc 'config/config_<ten_ban>.yaml'
MODELS = None            # None = lấy danh sách từ run.models trong config
                         # hoặc ['lasso', 'elasticnet']

from src.pipeline.Pipeline import HousePricePipeline, available_models
print('Model dùng được trên máy này:', available_models())

## Cách 1 — chạy một lệnh

Giống hệt `python main.py --member <ten>`. Xong là có `submission.csv`.

In [ ]:
out = HousePricePipeline(CONFIG, MEMBER).run(MODELS)
print('\nFile nộp Kaggle:', out)

---

## Cách 2 — chạy từng bước

Phần dưới làm đúng những gì `run()` làm, nhưng tách ra để xem số liệu ở giữa.
Dùng khi cần gỡ lỗi, hoặc lấy bảng và biểu đồ cho báo cáo.

### Bước 1: Clean → Feature → Preprocess

In [ ]:
pipe = HousePricePipeline(CONFIG, MEMBER)
X, y, X_test, test_ids = pipe.prepare()

print('X      :', X.shape)
print('X_test :', X_test.shape)
print('y (log):', round(y.min(), 3), '→', round(y.max(), 3))
assert list(X.columns) == list(X_test.columns), 'train và test lệch cột'
print('Train và test cùng bộ cột: OK')
X.head()

### Bước 2: cross-validation từng model

`oof_predict` chia 5 fold. Mỗi dòng train được dự đoán bởi model **chưa từng
thấy dòng đó**, nên con số này đáng tin hơn một lần chia train/test.
Với 1460 dòng, một lần chia cho sai số may rủi tới 0.01 RMSE.

In [ ]:
from src.evaluation.Evaluation import Evaluator
from src.pipeline.Pipeline import MODELS as MODEL_REGISTRY

names = MODELS or pipe.cfg.get('run', {}).get('models') or available_models()
names = [n for n in names if n in MODEL_REGISTRY]

results = {}
for n in names:
    results[n] = Evaluator.oof_predict(
        lambda k=n: MODEL_REGISTRY[k](pipe.cfg), X, y, X_test,
        n_folds=pipe.cfg['split']['n_folds'], seed=pipe.cfg['project']['seed'])
    r = results[n]
    print(f"{n:14s} rmse_oof={r['rmse_oof']:.5f}   cv={r['rmse_mean']:.5f} ± {r['rmse_std']:.5f}")

In [ ]:
Evaluator.compare(results)

### Bước 3: so sánh các cách kết hợp model

Bảng này là nội dung chính cho phần ensemble của báo cáo.

Lưu ý khi đọc: model nào bị gán trọng số gần 0 nghĩa là nó không thêm thông tin
so với các model còn lại, thường vì sai theo cùng một kiểu.

In [ ]:
from src.ensemble.Ensemble import compare_methods

oof = {k: v['oof'] for k, v in results.items()}
compare_methods(oof, y) if len(oof) > 1 else print('Cần ít nhất 2 model để trộn')

### Bước 4: biểu đồ chẩn đoán

Phần dư nên rải đều quanh 0. Nếu thấy hình phễu (càng về giá cao càng lệch)
thì model còn thiếu đặc trưng cho nhà đắt.

In [ ]:
from src.utils.data_loader import get_path

best = min(results, key=lambda k: results[k]['rmse_oof'])
print('Model tốt nhất:', best, '| rmse_oof =', round(results[best]['rmse_oof'], 5))

figs = get_path(pipe.cfg, 'figures')
Evaluator.plot_pred_vs_actual(y, oof[best], fig_dir=figs, fname=f'pred_vs_actual_{best}.png')
Evaluator.plot_residuals(y, oof[best], fig_dir=figs, fname=f'residuals_{best}.png')

### Bước 5: đặc trưng nào quan trọng

In [ ]:
model = MODEL_REGISTRY[best](pipe.cfg).fit(X, y)
try:
    imp = model.feature_importance(X.columns)
    Evaluator.plot_feature_importance(imp, k=25, fig_dir=figs,
                                      fname=f'importance_{best}.png')
    display(imp.head(20).to_frame('importance'))
except AttributeError as e:
    print('Model này không báo được độ quan trọng:', e)

---

## Sau khi chạy

- [ ] Nộp `SUBMIT-summarize_report/submission.csv` lên Kaggle
- [ ] Ghi một dòng vào `reports/results_summary.md`: ngày, người, model, RMSE OOF, điểm Kaggle
- [ ] Đối chiếu: điểm Kaggle lệch dưới 0.01 so với RMSE OOF là bình thường.
      Lệch nhiều hơn thì nghi có rò rỉ dữ liệu, phải tìm trước khi làm tiếp.

**Độ lệch chuẩn giữa các fold khoảng 0.007.** Mọi cải thiện nhỏ hơn con số đó
chưa thể kết luận là thật — muốn chắc thì chạy lại với `seed` khác trong config.